# SNP Feature Extraction — Top 10,000 SNPs Per Antibiotic
Trains ExtraTrees on the full SNP-only dataset for each antibiotic,  
ranks all SNPs by feature importance, and saves a **reduced SNP matrix**  
(Genome_ID + top 10,000 SNP columns) ready for merging with genes and k-mers.

In [1]:
# ── CELL 1: Imports ────────────────────────────────────────────────────────
import os
import pandas as pd
import numpy as np
import sklearn
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import ExtraTreesClassifier

print(f'sklearn  version : {sklearn.__version__}')
print(f'xgboost  version : {xgb.__version__}')
print(f'pandas   version : {pd.__version__}')
print(f'numpy    version : {np.__version__}')

sklearn  version : 1.4.2
xgboost  version : 3.1.3
pandas   version : 2.2.2
numpy    version : 1.26.4


In [2]:
# ── CELL 2: Configuration ──────────────────────────────────────────────────
# ⚠ Update DATA_DIR and OUTPUT_DIR to match your folder structure

DATA_DIR   = 'C:/Users/USER/Downloads/E_coli/datasets/final_merged_outputs'  # folder with SNPS_ONLY files
OUTPUT_DIR = 'C:/Users/USER/Downloads/E_coli/datasets/reduced_snps'          # where to save extracted SNP matrices

os.makedirs(OUTPUT_DIR, exist_ok=True)

TOP_N_SNPS   = 5000
RANDOM_STATE = 42

# Antibiotic prefix → label column name in your CSV
# ⚠ Adjust label column names if yours differ
ANTIBIOTICS = {
    'amoxicillin': 'Phenotype_Label',
    'coxime'     : 'Phenotype_Label',
    'doxy'       : 'Phenotype_Label',
    'levo'       : 'Phenotype_Label',
    'nali'       : 'Phenotype_Label',
    'norflo'     : 'Phenotype_Label',
    'strep'      : 'Phenotype_Label',
    'tetra'      : 'Phenotype_Label',
}

ANTIBIOTIC_FULL = {
    'amoxicillin': 'Amoxicillin/Clavulanic Acid',
    'coxime'     : 'Cefuroxime',
    'doxy'       : 'Doxycycline',
    'levo'       : 'Levofloxacin',
    'nali'       : 'Nalidixic Acid',
    'norflo'     : 'Norfloxacin',
    'strep'      : 'Streptomycin',
    'tetra'      : 'Tetracycline',
}

print(f'Output directory  : {OUTPUT_DIR}')
print(f'Top N SNPs to keep: {TOP_N_SNPS:,}')
print(f'Antibiotics       : {len(ANTIBIOTICS)}')

Output directory  : C:/Users/USER/Downloads/E_coli/datasets/reduced_snps
Top N SNPs to keep: 5,000
Antibiotics       : 8


In [3]:
# ── CELL 3: Extraction Loop ────────────────────────────────────────────────
# For each antibiotic:
#   1. Load SNPS_ONLY file
#   2. Train ExtraTrees to rank SNPs by importance
#   3. Save: (a) full importance ranking CSV
#            (b) reduced SNP matrix with only top 10,000 SNPs + Genome_ID

extraction_summary = []

for ab_prefix, label_col in ANTIBIOTICS.items():

    filepath = os.path.join(DATA_DIR, f'{ab_prefix}_SNPS_ONLY.csv')
    ab_full  = ANTIBIOTIC_FULL[ab_prefix]

    print(f'\n{"="*65}')
    print(f'  {ab_full}')
    print(f'{"="*65}')

    if not os.path.exists(filepath):
        print(f'  ⚠ File not found — skipping: {filepath}')
        extraction_summary.append({'Antibiotic': ab_full, 'Status': 'FILE NOT FOUND'})
        continue

    # --- Load ---
    df = pd.read_csv(filepath)
    df['Genome_ID'] = df['Genome_ID'].astype(str)

    if label_col not in df.columns:
        # Try to auto-detect label column
        candidates = [c for c in df.columns if 'phenotype' in c.lower() or 'label' in c.lower()]
        if candidates:
            label_col = candidates[0]
            print(f'  ℹ Auto-detected label column: {label_col}')
        else:
            print(f'  ⚠ Label column "{label_col}" not found. Columns: {df.columns.tolist()[:8]}...')
            extraction_summary.append({'Antibiotic': ab_full, 'Status': 'LABEL NOT FOUND'})
            continue

    genome_ids   = df['Genome_ID']
    y            = df[label_col]
    X            = df.drop(columns=['Genome_ID', label_col])
    feature_names = X.columns.tolist()

    print(f'  Samples         : {len(X)}')
    print(f'  Total SNPs      : {len(feature_names):,}')
    print(f'  Susceptible (0) : {(y==0).sum()}  |  Resistant (1): {(y==1).sum()}')

    actual_top_n = min(TOP_N_SNPS, len(feature_names))
    if actual_top_n < TOP_N_SNPS:
        print(f'  ⚠ Only {len(feature_names):,} SNPs available — keeping all')

    # --- Train ExtraTrees ---
    print(f'  Training ExtraTrees (200 trees)...')
    et = ExtraTreesClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    et.fit(X, y)

    # --- Rank by importance ---
    importance_df = pd.DataFrame({
        'Feature'   : feature_names,
        'Importance': et.feature_importances_
    }).sort_values('Importance', ascending=False).reset_index(drop=True)
    importance_df['Rank'] = importance_df.index + 1

    top_snps = importance_df.head(actual_top_n)['Feature'].tolist()

    print(f'  Top 10 SNPs by importance:')
    for _, row in importance_df.head(10).iterrows():
        print(f'    {int(row["Rank"]):>5}. {row["Feature"]:45s}  {row["Importance"]:.8f}')

    # --- Save importance ranking ---
    rank_path = os.path.join(OUTPUT_DIR, f'{ab_prefix}_snp_ranking.csv')
    importance_df.to_csv(rank_path, index=False)
    print(f'  ✓ Saved ranking  : {rank_path}')

    # --- Save reduced SNP matrix (Genome_ID + top N SNP columns ONLY, NO label) ---
    reduced_df = pd.concat([genome_ids, X[top_snps]], axis=1)
    reduced_path = os.path.join(OUTPUT_DIR, f'{ab_prefix}_snps_top{actual_top_n}.csv')
    reduced_df.to_csv(reduced_path, index=False)
    print(f'  ✓ Saved reduced  : {reduced_path}  → shape {reduced_df.shape}')

    extraction_summary.append({
        'Antibiotic'    : ab_full,
        'Total_SNPs'    : len(feature_names),
        'Kept_SNPs'     : actual_top_n,
        'Top_SNP'       : importance_df.iloc[0]['Feature'],
        'Top_Importance': round(importance_df.iloc[0]['Importance'], 8),
        'Status'        : 'OK',
        'Reduced_File'  : os.path.basename(reduced_path),
    })

# --- Summary table ---
summary_df = pd.DataFrame(extraction_summary)
print(f'\n{"="*65}')
print('EXTRACTION SUMMARY')
print(f'{"="*65}')
print(summary_df.to_string(index=False))

summary_df.to_csv(os.path.join(OUTPUT_DIR, 'snp_extraction_summary.csv'), index=False)
print('✓ Summary saved to: {OUTPUT_DIR}/snp_extraction_summary.csv')
print('Next step: use the reduced files in OUTPUT_DIR to merge with genes and k-mers')


  Amoxicillin/Clavulanic Acid
  Samples         : 571
  Total SNPs      : 35,476
  Susceptible (0) : 319  |  Resistant (1): 252
  Training ExtraTrees (200 trees)...
  Top 10 SNPs by importance:
        1. NC_000913_3165735_C>A                          0.00606569
        2. NC_000913_3832753_G>A                          0.00575232
        3. NC_000913_3296252_C>T                          0.00382839
        4. NC_000913_3295235_T>C                          0.00375509
        5. NC_000913_1869977_T>A                          0.00357223
        6. NC_000913_2803237_G>A                          0.00338885
        7. NC_000913_2296932_T>A                          0.00312808
        8. NC_000913_4117497_T>C                          0.00306054
        9. NC_000913_964580_A>T                           0.00292211
       10. NC_000913_794685_A>G                           0.00281243
  ✓ Saved ranking  : C:/Users/USER/Downloads/E_coli/datasets/reduced_snps\amoxicillin_snp_ranking.csv
  ✓ Saved red